# Using Pre-Trained Model

In [1]:
!pip install -q transformers sentencepiece nltk rouge-score

from transformers import MarianMTModel, MarianTokenizer
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

english_sentences = [
    "hello", "how are you", "i am fine", "what is your name", "nice to meet you",
    "i love machine learning", "do you like pizza", "good morning", "thank you", "see you later"
]
french_sentences = [
    "bonjour", "comment ça va", "je vais bien", "quel est ton nom", "ravi de vous rencontrer",
    "j'aime l'apprentissage automatique", "aimes-tu la pizza", "bonjour", "merci", "à plus tard"
]

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

results = []
for i, src in enumerate(english_sentences):
    inputs = tokenizer(src, return_tensors="pt", padding=True)
    translated = model.generate(**inputs)
    tgt = tokenizer.decode(translated[0], skip_special_tokens=True)

    reference = [nltk.word_tokenize(french_sentences[i])]
    prediction = nltk.word_tokenize(tgt)
    bleu = sentence_bleu(reference, prediction)
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True).score(
        ' '.join(reference[0]), ' '.join(prediction))

    results.append({
        "English": src,
        "French (Reference)": french_sentences[i],
        "Predicted French": tgt,
        "BLEU": round(bleu, 4),
        "ROUGE-1": round(rouge['rouge1'].fmeasure, 4),
        "ROUGE-2": round(rouge['rouge2'].fmeasure, 4)
    })

import pandas as pd
pd.DataFrame(results)

  Preparing metadata (setup.py) ... done


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

,English,French (Reference),Predicted French,BLEU,ROUGE-1,ROUGE-2
0,hello,bonjour,Bonjour.,0.0,1.0000,0.0000
1,how are you,comment ça va,comment allez-vous,0.0,0.3333,0.0000
2,i am fine,je vais bien,Je vais bien.,0.0,1.0000,1.0000
3,what is your name,quel est ton nom,quel est votre nom,0.0,0.7500,0.3333
4,nice to meet you,ravi de vous rencontrer,Ravie de vous rencontrer.,0.0,1.0000,1.0000
5,i love machine learning,j'aime l'apprentissage automatique,J'adore l'apprentissage automatique,0.0,0.8000,0.5000
6,do you like pizza,aimes-tu la pizza,Tu aimes les pizzas ?,0.0,0.7500,0.0000
7,good morning,bonjour,Bonjour.,0.0,1.0000,0.0000
8,thank you,merci,Je vous remercie.,0.0,0.0000,0.0000
9,see you later,à plus tard,A tout à l'heure.,0.0,0.0000,0.0000


In [2]:
# Step 1: Install and import libraries
!pip install -q datasets nltk rouge-score

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import tensorflow as tf
import nltk
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
nltk.download('punkt')

# Step 2: Prepare a small dataset (English-French)
english_sentences = [
    "hello", "how are you", "i am fine", "what is your name", "nice to meet you",
    "i love machine learning", "do you like pizza", "good morning", "thank you", "see you later"
]

french_sentences = [
    "bonjour", "comment ça va", "je vais bien", "quel est ton nom", "ravi de vous rencontrer",
    "j'aime l'apprentissage automatique", "aimes-tu la pizza", "bonjour", "merci", "à plus tard"
]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
len(english_sentences)

10

In [4]:


# Step 3: Tokenization
def tokenize(sentences, num_words=10000):
    tokenizer = Tokenizer(num_words=num_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(sentences)
    tensor = tokenizer.texts_to_sequences(sentences)
    return tokenizer, pad_sequences(tensor, padding='post')

en_tokenizer, en_tensor = tokenize(english_sentences)
fr_tokenizer, fr_tensor = tokenize(french_sentences)
input_vocab_size = len(en_tokenizer.word_index) + 1
target_vocab_size = len(fr_tokenizer.word_index) + 1


In [6]:
en_tokenizer.word_index

{'<OOV>': 1,
 'you': 2,
 'i': 3,
 'hello': 4,
 'how': 5,
 'are': 6,
 'am': 7,
 'fine': 8,
 'what': 9,
 'is': 10,
 'your': 11,
 'name': 12,
 'nice': 13,
 'to': 14,
 'meet': 15,
 'love': 16,
 'machine': 17,
 'learning': 18,
 'do': 19,
 'like': 20,
 'pizza': 21,
 'good': 22,
 'morning': 23,
 'thank': 24,
 'see': 25,
 'later': 26}

In [7]:
en_tensor

array([[ 4,  0,  0,  0],
       [ 5,  6,  2,  0],
       [ 3,  7,  8,  0],
       [ 9, 10, 11, 12],
       [13, 14, 15,  2],
       [ 3, 16, 17, 18],
       [19,  2, 20, 21],
       [22, 23,  0,  0],
       [24,  2,  0,  0],
       [25,  2, 26,  0]], dtype=int32)

In [8]:
target_vocab_size

28

In [9]:
fr_tokenizer.word_index

{'<OOV>': 1,
 'bonjour': 2,
 'comment': 3,
 'ça': 4,
 'va': 5,
 'je': 6,
 'vais': 7,
 'bien': 8,
 'quel': 9,
 'est': 10,
 'ton': 11,
 'nom': 12,
 'ravi': 13,
 'de': 14,
 'vous': 15,
 'rencontrer': 16,
 "j'aime": 17,
 "l'apprentissage": 18,
 'automatique': 19,
 'aimes': 20,
 'tu': 21,
 'la': 22,
 'pizza': 23,
 'merci': 24,
 'à': 25,
 'plus': 26,
 'tard': 27}

In [10]:
# Step 4: Define transformer model
class Transformer(tf.keras.Model):
    def __init__(self, input_vocab, target_vocab, d_model, num_heads, dff, pe_input, pe_target):
        super().__init__()
        self.encoder_embedding = tf.keras.layers.Embedding(input_vocab, d_model)
        self.decoder_embedding = tf.keras.layers.Embedding(target_vocab, d_model)
        self.pos_encoding_input = self.positional_encoding(pe_input, d_model)
        self.pos_encoding_target = self.positional_encoding(pe_target, d_model)

        self.enc_layer = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.dec_layer = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.dense_proj = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])
        self.final_dense = tf.keras.layers.Dense(target_vocab)

    def positional_encoding(self, max_len, dm):
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(dm)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(dm))
        angle_rads = pos * angle_rates
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        return tf.cast(angle_rads, dtype=tf.float32)

    def call(self, inputs, training=False):
        inp, tar = inputs
        enc = self.encoder_embedding(inp) + self.pos_encoding_input[:tf.shape(inp)[1]]
        dec = self.decoder_embedding(tar) + self.pos_encoding_target[:tf.shape(tar)[1]]
        enc_output = self.enc_layer(enc, enc)
        dec_output = self.dec_layer(dec, enc_output)
        final = self.dense_proj(dec_output)
        return self.final_dense(final)


In [11]:


# Step 5: Compile and train
model = Transformer(input_vocab=input_vocab_size, target_vocab=target_vocab_size,
                    d_model=64, num_heads=4, dff=128, pe_input=en_tensor.shape[1], pe_target=fr_tensor.shape[1])
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
model.fit([en_tensor, fr_tensor[:, :-1]], fr_tensor[:, 1:], epochs=20, batch_size=32)


Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - accuracy: 0.0000e+00 - loss: 3.3306
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.4333 - loss: 3.2668
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - accuracy: 0.4333 - loss: 3.1652
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.4333 - loss: 3.0055
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.4333 - loss: 2.7805
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.4333 - loss: 2.5750
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.4333 - loss: 2.7348
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.4333 - loss: 2.6575
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.4333 - loss: 2.4962
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - accuracy: 0.4333 - loss: 2.4643
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4333 - loss: 2.4972
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.4333 - loss

In [12]:

# Step 6: Translation function
def translate(sentence):
    seq = en_tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=en_tensor.shape[1], padding='post')
    decoder_input = np.zeros((1, fr_tensor.shape[1]), dtype=np.int32)
    decoder_input[0][0] = fr_tokenizer.word_index.get('<sos>', 1)

    for i in range(1, fr_tensor.shape[1]):
        output = model([padded, decoder_input[:, :-1]], training=False)
        pred_id = tf.argmax(output[0, i-1]).numpy()
        decoder_input[0][i] = pred_id
        if pred_id == fr_tokenizer.word_index.get('<eos>'):
            break

    return ' '.join([fr_tokenizer.index_word.get(idx, '') for idx in decoder_input[0] if idx != 0])


In [13]:

# Step 7: Evaluation
results = []
for i in range(10):
    ref = [nltk.word_tokenize(french_sentences[i].replace('<sos>', '').replace('<eos>', '').strip())]
    pred = nltk.word_tokenize(translate(english_sentences[i]))
    bleu = sentence_bleu(ref, pred)
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True).score(' '.join(ref[0]), ' '.join(pred))
    results.append((english_sentences[i], french_sentences[i].replace('<sos>', '').replace('<eos>', '').strip(), ' '.join(pred), bleu, rouge['rouge1'].fmeasure, rouge['rouge2'].fmeasure))

import pandas as pd
pd.DataFrame(results, columns=["English", "French (Reference)", "Predicted French", "BLEU", "ROUGE-1", "ROUGE-2"])

,English,French (Reference),Predicted French,BLEU,ROUGE-1,ROUGE-2
0,hello,bonjour,< OOV >,0,0.0,0.0
1,how are you,comment ça va,< OOV >,0,0.0,0.0
2,i am fine,je vais bien,< OOV >,0,0.0,0.0
3,what is your name,quel est ton nom,< OOV >,0,0.0,0.0
4,nice to meet you,ravi de vous rencontrer,< OOV >,0,0.0,0.0
5,i love machine learning,j'aime l'apprentissage automatique,< OOV >,0,0.0,0.0
6,do you like pizza,aimes-tu la pizza,< OOV >,0,0.0,0.0
7,good morning,bonjour,< OOV >,0,0.0,0.0
8,thank you,merci,< OOV >,0,0.0,0.0
9,see you later,à plus tard,< OOV >,0,0.0,0.0


In [ ]:
# Step 1: Install and import libraries
!pip install -q datasets nltk rouge-score

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import tensorflow as tf
import nltk
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
nltk.download('punkt')

# Step 2: Load a real translation dataset (English-French)
data = load_dataset("opus_books", "en-fr", split='train[:10000]')
english_sentences = [f"{x['translation']['en']}" for x in data]
french_sentences = [f"<sos> {x['translation']['fr']} <eos>" for x in data]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

In [ ]:
english_sentences[0]

'The Wanderer'

In [ ]:
french_sentences[0]

'<sos> Le grand Meaulnes <eos>'

In [ ]:
len(english_sentences)

10000

In [ ]:

# Step 3: Tokenization
def tokenize(sentences, num_words=10000):
    tokenizer = Tokenizer(num_words=num_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(sentences)
    tensor = tokenizer.texts_to_sequences(sentences)
    return tokenizer, pad_sequences(tensor, padding='post',maxlen=40)

en_tokenizer, en_tensor = tokenize(english_sentences)
fr_tokenizer, fr_tensor = tokenize(french_sentences)
input_vocab_size = len(en_tokenizer.word_index) + 1
target_vocab_size = len(fr_tokenizer.word_index) + 1


In [ ]:
en_tokenizer.word_index

{'<OOV>': 1,
 'the': 2,
 'to': 3,
 'and': 4,
 'of': 5,
 'i': 6,
 'a': 7,
 'in': 8,
 'was': 9,
 'her': 10,
 'he': 11,
 'she': 12,
 'it': 13,
 'that': 14,
 'had': 15,
 'his': 16,
 'not': 17,
 'you': 18,
 'with': 19,
 'as': 20,
 'at': 21,
 'for': 22,
 'on': 23,
 'be': 24,
 'my': 25,
 'but': 26,
 'is': 27,
 'me': 28,
 'have': 29,
 'him': 30,
 'by': 31,
 'which': 32,
 'all': 33,
 'from': 34,
 'were': 35,
 'so': 36,
 'this': 37,
 'they': 38,
 'mr': 39,
 'could': 40,
 'no': 41,
 'been': 42,
 'one': 43,
 'what': 44,
 'would': 45,
 'said': 46,
 'when': 47,
 'we': 48,
 'very': 49,
 "'": 50,
 'there': 51,
 'an': 52,
 'their': 53,
 'them': 54,
 'your': 55,
 'elizabeth': 56,
 'or': 57,
 'who': 58,
 'if': 59,
 'are': 60,
 'will': 61,
 'then': 62,
 'more': 63,
 'do': 64,
 'little': 65,
 'up': 66,
 'out': 67,
 'such': 68,
 'now': 69,
 'must': 70,
 'miss': 71,
 'mrs': 72,
 'time': 73,
 'some': 74,
 'much': 75,
 'am': 76,
 'did': 77,
 'only': 78,
 'than': 79,
 'before': 80,
 'room': 81,
 'darcy': 82,
 '

In [ ]:
target_vocab_size

17739

In [ ]:

# Step 4: Define transformer model
class Transformer(tf.keras.Model):
    def __init__(self, input_vocab, target_vocab, d_model, num_heads, dff, pe_input, pe_target):
        super().__init__()
        self.encoder_embedding = tf.keras.layers.Embedding(input_vocab, d_model)
        self.decoder_embedding = tf.keras.layers.Embedding(target_vocab, d_model)
        self.pos_encoding_input = self.positional_encoding(pe_input, d_model)
        self.pos_encoding_target = self.positional_encoding(pe_target, d_model)

        self.enc_layer = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.dec_layer = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.dense_proj = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])
        self.final_dense = tf.keras.layers.Dense(target_vocab)

    def positional_encoding(self, max_len, dm):
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(dm)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(dm))
        angle_rads = pos * angle_rates
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        return tf.cast(angle_rads, dtype=tf.float32)

    def call(self, inputs, training=False):
        inp, tar = inputs
        enc = self.encoder_embedding(inp) + self.pos_encoding_input[:tf.shape(inp)[1]]
        dec = self.decoder_embedding(tar) + self.pos_encoding_target[:tf.shape(tar)[1]]
        enc_output = self.enc_layer(enc, enc)
        dec_output = self.dec_layer(dec, enc_output)
        final = self.dense_proj(dec_output)
        return self.final_dense(final)


In [ ]:
en_tensor.shape

(10000, 40)

In [ ]:

# Step 5: Compile and train
model = Transformer(input_vocab=input_vocab_size, target_vocab=target_vocab_size,
                    d_model=64, num_heads=4, dff=128, pe_input=en_tensor.shape[1], pe_target=fr_tensor.shape[1])
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
model.fit([en_tensor, fr_tensor[:, :-1]], fr_tensor[:, 1:], epochs=100, batch_size=32)


Epoch 1/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.4581 - loss: 4.9592
Epoch 2/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.4837 - loss: 3.4979
Epoch 3/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.5183 - loss: 3.2659
Epoch 4/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.5211 - loss: 3.1655
Epoch 5/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.5289 - loss: 3.0185
Epoch 6/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.5292 - loss: 2.9383
Epoch 7/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.5403 - loss: 2.8134
Epoch 8/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.5407 - loss: 2.7634
Epoch 9/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.5445 - loss: 2.7079
Epoch 10/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.5495 - loss: 2.6482
Epoch 11/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.5436 - loss: 2.6511
Epoch 12/100
313/313 ━━━━━━━━

In [ ]:
fr_tensor.shape

(10000, 40)

In [ ]:
en_tokenizer.word_index

{'<OOV>': 1,
 'the': 2,
 'to': 3,
 'and': 4,
 'of': 5,
 'i': 6,
 'a': 7,
 'in': 8,
 'was': 9,
 'her': 10,
 'he': 11,
 'she': 12,
 'it': 13,
 'that': 14,
 'had': 15,
 'his': 16,
 'not': 17,
 'you': 18,
 'with': 19,
 'as': 20,
 'at': 21,
 'for': 22,
 'on': 23,
 'be': 24,
 'my': 25,
 'but': 26,
 'is': 27,
 'me': 28,
 'have': 29,
 'him': 30,
 'by': 31,
 'which': 32,
 'all': 33,
 'from': 34,
 'were': 35,
 'so': 36,
 'this': 37,
 'they': 38,
 'mr': 39,
 'could': 40,
 'no': 41,
 'been': 42,
 'one': 43,
 'what': 44,
 'would': 45,
 'said': 46,
 'when': 47,
 'we': 48,
 'very': 49,
 "'": 50,
 'there': 51,
 'an': 52,
 'their': 53,
 'them': 54,
 'your': 55,
 'elizabeth': 56,
 'or': 57,
 'who': 58,
 'if': 59,
 'are': 60,
 'will': 61,
 'then': 62,
 'more': 63,
 'do': 64,
 'little': 65,
 'up': 66,
 'out': 67,
 'such': 68,
 'now': 69,
 'must': 70,
 'miss': 71,
 'mrs': 72,
 'time': 73,
 'some': 74,
 'much': 75,
 'am': 76,
 'did': 77,
 'only': 78,
 'than': 79,
 'before': 80,
 'room': 81,
 'darcy': 82,
 '

In [ ]:
len(en_tokenizer.word_index)

12241

In [ ]:

# Step 6: Translation function
def translate(sentence):
    seq = en_tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=en_tensor.shape[1], padding='post')
    decoder_input = np.zeros((1, fr_tensor.shape[1]), dtype=np.int32)
    decoder_input[0][0] = fr_tokenizer.word_index.get('<sos>', 1)

    for i in range(1, fr_tensor.shape[1]):
        output = model([padded, decoder_input[:, :-1]], training=False)
        pred_id = tf.argmax(output[0, i-1]).numpy()
        decoder_input[0][i] = pred_id
        if pred_id == fr_tokenizer.word_index.get('<eos>'):
            break

    return ' '.join([fr_tokenizer.index_word.get(idx, '') for idx in decoder_input[0] if idx != 0])

# Step 7: Evaluation
results = []
for i in range(10):
    ref = [nltk.word_tokenize(french_sentences[i].replace('<sos>', '').replace('<eos>', '').strip())]
    pred = nltk.word_tokenize(translate(english_sentences[i]))
    bleu = sentence_bleu(ref, pred)
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True).score(' '.join(ref[0]), ' '.join(pred))
    results.append((english_sentences[i], french_sentences[i].replace('<sos>', '').replace('<eos>', '').strip(), ' '.join(pred), bleu, rouge['rouge1'].fmeasure, rouge['rouge2'].fmeasure))

import pandas as pd
pd.DataFrame(results, columns=["English", "French (Reference)", "Predicted French", "BLEU", "ROUGE-1", "ROUGE-2"])

/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

,English,French (Reference),Predicted French,BLEU,ROUGE-1,ROUGE-2
0,The Wanderer,Le grand Meaulnes,< OOV > eos,0.000000e+00,0.000000,0.000000
1,Alain-Fournier,Alain-Fournier,< OOV > eos,0.000000e+00,0.000000,0.000000
2,First Part,PREMIÈRE PARTIE,< OOV > eos,0.000000e+00,0.000000,0.000000
3,I,CHAPITRE PREMIER,< OOV > eos,0.000000e+00,0.000000,0.000000
4,THE BOARDER,LE PENSIONNAIRE,< OOV > eos,0.000000e+00,0.000000,0.000000
5,He arrived at our home on a Sunday of November...,Il arriva chez nous un dimanche de novembre 189-…,< OOV > à la route de la route de la route eos,9.594503e-232,0.105263,0.000000
6,"I still say 'our home,' although the house no ...","Je continue à dire « chez nous », bien que la ...",< OOV > que nous < OOV > que je < OOV > de < O...,1.051835e-231,0.260870,0.000000
7,We left that part of the country nearly fiftee...,Nous avons quitté le pays depuis bientôt quinz...,< OOV > que nous avons été bien de la < OOV > ...,1.037713e-231,0.200000,0.071429
8,We were living in the building of the Higher E...,Nous habitions les bâtiments du Cours Supérieu...,< OOV > dans la route de la route de galais eos,9.788429e-232,0.090909,0.000000
9,"My father, whom I used to call M. Seurel as di...","Mon père, que j’appelais M. Seurel, comme les ...",< OOV > de galais me dit la route et je me dit...,1.000369e-231,0.138889,0.000000


In [ ]:
import os
from google.colab import userdata



# Set environment variables for Kaggle API
os.environ['KAGGLE_USERNAME'] ="mohanrajmit"
os.environ['KAGGLE_KEY'] = "3bbbe7ee9dbe85615bfec6346e22510a"

print("Kaggle credentials set as environment variables.")

Kaggle credentials set as environment variables.


In [ ]:
#!/bin/bash
!kaggle datasets download bagavathypriya/english-to-tamil-data

Dataset URL: https://www.kaggle.com/datasets/bagavathypriya/english-to-tamil-data
License(s): unknown
  0% 0.00/9.29k [00:00<?, ?B/s]
100% 9.29k/9.29k [00:00<00:00, 11.6MB/s]


In [ ]:
!unzip /content/english-to-tamil-data.zip

Archive:  /content/english-to-tamil-data.zip
  inflating: tam.txt                 


In [ ]:
english_sentences = []
tamil_sentences = []

with open('/content/tam.txt', 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            english_sentences.append(parts[0])
            tamil_sentences.append(parts[1])

print(f"Number of English sentences: {len(english_sentences)}")
print(f"Number of Tamil sentences: {len(tamil_sentences)}")

Number of English sentences: 201
Number of Tamil sentences: 201


In [ ]:
english_sentences[0]

'I slept.'

In [ ]:
tamil_sentences[0]

'நான் தூங்கினேன்.'

In [ ]:
en_tokenizer, en_tensor = tokenize(english_sentences)
tamil_tokenizer, tamil_tensor = tokenize(tamil_sentences)
input_vocab_size = len(en_tokenizer.word_index) + 1
target_vocab_size = len(tamil_tokenizer.word_index) + 1

In [ ]:
tamil_tokenizer.word_index

{'<OOV>': 1,
 'நான்': 2,
 'அவள்': 3,
 'அவன்': 4,
 'டாம்': 5,
 'ஒரு': 6,
 'என்று': 7,
 'அவனுக்கு': 8,
 'தெரியும்': 9,
 'என்னிடம்': 10,
 'நீ': 11,
 'வேண்டும்': 12,
 'எனக்கு': 13,
 'நிறைய': 14,
 'என்ன': 15,
 'இது': 16,
 'இந்த': 17,
 'யார்': 18,
 'போய்': 19,
 'இருக்கிறது': 20,
 'அந்த': 21,
 'விரும்புகிறேன்': 22,
 'எப்பொழுது': 23,
 'நாங்கள்': 24,
 'உன்னிடம்': 25,
 'என்': 26,
 'பயம்': 27,
 'அவர்கள்': 28,
 'அவனைக்': 29,
 'அவளிடம்': 30,
 'அதை': 31,
 'ஓட': 32,
 'வந்தான்': 33,
 'எப்படி': 34,
 'சொன்னான்': 35,
 'எனக்குத்': 36,
 'பணம்': 37,
 'செல்ல': 38,
 'செய்து': 39,
 'வெளியே': 40,
 'மேரியுடன்': 41,
 'அமைதியாக': 42,
 'சிரித்தாள்': 43,
 'எங்கே': 44,
 'இரு': 45,
 'பார்க்கிறேன்': 46,
 'உன்': 47,
 'வந்து': 48,
 'பார்': 49,
 'ஆரம்பித்தான்': 50,
 'இருக்கிறார்கள்': 51,
 'என்னால்': 52,
 'சாப்பிட': 53,
 'முடியும்': 54,
 'வா': 55,
 'எங்களுக்கு': 56,
 'உதவி': 57,
 'இன்னும்': 58,
 'இப்பொழுது': 59,
 'போக': 60,
 'எங்களுடைய': 61,
 'என்னுடைய': 62,
 'முன்னால்': 63,
 'மூன்று': 64,
 'என்பது': 65,
 'போகத்': 66,
 'நட

In [ ]:
en_tensor.shape

(201, 40)

In [ ]:
tamil_tensor.shape

(201, 40)

In [ ]:

# Step 5: Compile and train
model = Transformer(input_vocab=input_vocab_size, target_vocab=target_vocab_size,
                    d_model=64, num_heads=4, dff=128, pe_input=en_tensor.shape[1], pe_target=tamil_tensor.shape[1])
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
model.fit([en_tensor, tamil_tensor[:, :-1]],tamil_tensor[:, 1:], epochs=1000, batch_size=32)


Epoch 1/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 397ms/step - accuracy: 0.5986 - loss: 6.0666
Epoch 2/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9203 - loss: 1.8679  
Epoch 3/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9175 - loss: 1.7860 
Epoch 4/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9178 - loss: 1.0007 
Epoch 5/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9196 - loss: 0.9098 
Epoch 6/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9183 - loss: 0.8219 
Epoch 7/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9179 - loss: 0.8009 
Epoch 8/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9197 - loss: 0.7812 
Epoch 9/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9189 - loss: 0.7821 
Epoch 10/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9210 - loss: 0.7615 
Epoch 11/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9183 - loss: 0.7799 
Epoch 12/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy

In [ ]:
# Step 6: Translation function
def translate_tamil(sentence):
    seq = en_tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=en_tensor.shape[1], padding='post')
    decoder_input = np.zeros((1, tamil_tensor.shape[1]), dtype=np.int32)
    decoder_input[0][0] = tamil_tokenizer.word_index.get('<sos>', 1)

    for i in range(1, tamil_tensor.shape[1]):
        output = model([padded, decoder_input[:, :-1]], training=False)
        pred_id = tf.argmax(output[0, i-1]).numpy()
        decoder_input[0][i] = pred_id
        if pred_id == tamil_tokenizer.word_index.get('<eos>'):
            break

    return ' '.join([tamil_tokenizer.index_word.get(idx, '') for idx in decoder_input[0] if idx != 0])

# Example usage with an unknown English sentence
unknown_english_sentence = "how are you?"
translated_tamil_sentence = translate_tamil(unknown_english_sentence)
print(f"Original English: {unknown_english_sentence}")
print(f"Translated Tamil: {translated_tamil_sentence}")

Original English: how are you?
Translated Tamil: <OOV> விரும்புகிறேன்
